# Kickstarter Projects — Data Analysis

## 1. Introduction to the Dataset

### 1.1 Source and Context of the Data

- **Dataset File:** `dateSet/ks-projects-201801.csv`
- **Source:** Publicly **Kickstarter Projects** dataset published on Kaggle by Mickaël Mouillé , collected directly from the Kickstarter platform as of **January 2018**.
- **Domain & Platform Context:** [Kickstarter](https://www.kickstarter.com/) is one of the world's largest reward-based crowdfunding platforms for creative, artistic, and technological projects . Kickstarter operates on an **all-or-nothing funding model**: creators set a monetary funding goal and a fixed campaign deadline. Backers pledge funds in exchange for rewards, and funds are collected only if total pledges reach or exceed the stated goal before the deadline.
- **Scope & Size:**
  - **378,661** unique crowdfunding campaigns across **15** columns.
  - **Time Period:** Launch dates span from **April 21, 2009**  through **January 2, 2018** , and deadlines extend through **March 3, 2018**.
  - **Global Coverage:** Projects span **15 main categories** , **14 currencies**, and **22 valid countries** (plus 3,797 records with an invalid `N,0"` country placeholder), with the United States (~77.3% of campaigns) and USD (~78.0%) representing the largest share.
- **Analytical Relevance:** We selected this dataset because it enables examining how pre-launch decisions made by creators and entrepreneurs relate to campaign success, while allowing comparisons across diverse domains and countries. Specifically, our analysis aims to identify the features that most strongly influence whether a campaign succeeds or fails, evaluate whether setting an overly high funding goal reduces the probability of success alongside the optimal goal range within each category, and assess whether shorter campaigns achieve higher success rates than longer ones as well as how launch timing (month and day of the week) affects backer counts and total funds raised. In addition, we investigate which categories exhibit the highest success rates and average pledge amounts, whether significant disparities exist across countries, and—among failed campaigns—how closely projects approached their funding goals versus raising only a small fraction of them, and whether this failure pattern varies across categories.

### 1.2 Description of Columns (Features) and Their Meanings

| # | Column Name | Data Type | Description & Meaning | Notes / Observed Characteristics |
|---:|---|---|---|---| 
| 1 | `ID` | Integer | Internal unique identifier assigned by Kickstarter to each project. | 378,661 unique values (primary key; non-predictive). |
| 2 | `name` | String (Text) | Title / name of the Kickstarter campaign. | 4 missing values; useful for text/NLP feature engineering (e.g., title length, keywords). |
| 3 | `category` | Categorical (String) | Specific subcategory describing the project (e.g., *Narrative Film*, *Tabletop Games*, *Product Design*). | 159 unique subcategories. |
| 4 | `main_category` | Categorical (String) | Broad top-level project domain (e.g., *Film & Video*, *Music*, *Publishing*, *Games*, *Technology*). | 15 unique main categories. |
| 5 | `currency` | Categorical (String) | Currency in which the creator's goal and pledges are denominated (ISO code, e.g., `USD`, `GBP`, `EUR`, `CAD`). | 14 unique currencies; `USD` accounts for 295,365 projects (~78%). |
| 6 | `deadline` | Date (`YYYY-MM-DD`) | Final closing date by which the campaign must reach its funding goal. | Ranges from `2009-05-03` to `2018-03-03`. |
| 7 | `goal` | Float (Numeric) | Target funding amount set by the creator in the project's **local currency**. | Ranges from 0.01 to 100,000,000 (not directly comparable across currencies). |
| 8 | `launched` | Datetime (`YYYY-MM-DD HH:MM:SS`) | Exact date and time when the campaign went live on Kickstarter. | Valid range `2009-04-21` to `2018-01-02`; 7 anomalous records contain `1970-01-01`. Can be combined with `deadline` to compute campaign duration. |
| 9 | `pledged` | Float (Numeric) | Total amount pledged by backers in the project's **local currency**. | Ranges from 0.00 to 20,338,986.27; post-launch outcome metric. |
| 10 | `state` | Categorical (String) | Final or current status of the campaign as of January 2018: `failed`, `successful`, `canceled`, `undefined`, `live`, or `suspended`. | 6 unique states: `failed` (52.22%), `successful` (35.38%), `canceled` (10.24%), `undefined` (0.94%), `live` (0.74%), `suspended` (0.49%). |
| 11 | `backers` | Integer (Numeric) | Total number of unique individuals who pledged money to the campaign. | Ranges from 0 to 219,382; post-launch outcome metric. |
| 12 | `country` | Categorical (String) | Two-letter ISO country code where the project creator is based (e.g., `US`, `GB`, `CA`, `AU`). | 22 valid countries plus 3,797 malformed entries recorded as `N,0"`. |
| 13 | `usd pledged` | Float (Numeric) | Amount pledged converted to US Dollars as originally computed by Kickstarter. | Contains 3,797 missing values (matching the `N,0"` country rows) and occasional currency conversion errors. |
| 14 | `usd_pledged_real` | Float (Numeric) | Amount pledged converted to US Dollars using historical exchange rates from the Fixer.io API. | No missing values; ranges from \$0.00 to \$20,338,986.27. Preferred over `usd pledged` for cross-currency analysis. |
| 15 | `usd_goal_real` | Float (Numeric) | Funding goal converted to US Dollars using historical exchange rates from the Fixer.io API. | No missing values; ranges from \$0.01 to \$166,361,390.71. Preferred over `goal` for cross-currency comparisons. |

### 1.3 Identification of Potential Target Variables

To achieve our research objectives and examine how creators' pre-launch decisions relate to campaign outcomes across categories and countries, we distinguish between the **target variables** (outcomes we measure or predict) and the **pre-launch feature variables** (decisions made before launch).

#### Target Variables (Campaign Outcomes)
- **`state` (`successful` vs. `failed`):** Our primary binary classification target. It enables identifying which pre-launch features most strongly influence whether a campaign succeeds or fails, testing whether high funding goals lower success probability, comparing success rates of shorter vs. longer campaigns, and measuring success-rate differences across categories and countries.
- **`usd_pledged_real`:** A secondary quantitative target representing total funds raised in standardized USD. It helps evaluate whether launch timing (month and day of the week) and pre-launch choices affect the total amount raised.
- **`backers`:** A secondary quantitative target representing the number of supporters. It helps assess how launch timing (month and day of the week) and campaign characteristics influence supporter volume.
- **Derived Target: Average Pledge per Backer (`usd_pledged_real / backers`, for `backers > 0`):** Helps identify which categories and countries achieve the highest average contribution per supporter.
- **Derived Target: Funding Ratio (`usd_pledged_real / usd_goal_real`):** Helps analyze failed campaigns by measuring how many came close to reaching their goal versus raising only a small fraction of it, and whether this pattern differs across categories.

#### Pre-Launch Feature Variables
- **`usd_goal_real`:** Helps evaluate whether setting an overly high funding goal reduces the probability of success and identify the optimal goal range within each category.
- **`launched` and `deadline`:** Used to compute campaign duration (`deadline - launched`) to test whether shorter campaigns succeed at higher rates than longer ones, and to extract launch month and day of the week to examine their effect on backer counts and total funds raised.
- **`main_category` and `category`:** Help determine which categories achieve the highest success rates and average pledge amounts, define optimal goal ranges per category, and compare failed-campaign funding patterns across categories.
- **`country` (and `currency`):** Help examine whether significant disparities in campaign success and funding performance exist across countries.
- **`name`:** Allows testing whether campaign title characteristics (such as length or keywords) contribute to predicting campaign success before launch.

## 2. Initial Data Loading & Verification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('dateSet/ks-projects-201801.csv')
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nTarget (state) distribution:')
print(pd.DataFrame({
    'count': df['state'].value_counts(),
    'percentage': (df['state'].value_counts(normalize=True) * 100).round(2)
}))